# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and processing the *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells* dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and discover the schema and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant dataset metadata and create a Dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset name:\n", metadata.name)
print("\nDescription:\n", metadata.description)
print("\nPublished:", getattr(metadata, 'datePublished', None))


## 2. Data Overview
Let's review available RecordSets, their `@id`s, fields and corresponding column information in the dataset.


In [ ]:
# List all RecordSets, their @ids, and field details

record_sets = dataset.record_sets
print(f"Number of RecordSets: {len(record_sets)}\n")

for record_set in record_sets:
    print(f"RecordSet name: {record_set.name}")
    print(f"\t@id: {record_set.id}")
    print(f"\tDescription: {getattr(record_set, 'description', None)}")
    print(f"\tFields:")
    for field in record_set.fields:
        print(f"\t - {field.name}\n\t   @id: {field.id}\n\t   Data type: {field.data_type}")
        if hasattr(field, 'column') and field.column is not None:
            print(f"\t   Column @id: {field.column.id if hasattr(field.column,'id') else field.column}")
    print("\n---\n")

### Example Preview: Inspect One RecordSet
The following displays a sample record from the first RecordSet for reference.

In [ ]:
# Show the first record for each record set (referenced by @id)
for record_set in dataset.record_sets:
    print(f"\nRecordSet: {record_set.name} (@id: {record_set.id})")
    for i, record in enumerate(dataset.records(record_set=record_set.id)):
        print(json.dumps(record, indent=2))
        break  # Just the first record


## 3. Data Extraction
We'll load data for **all record sets** into pandas DataFrames using their `@id` values.

You can reference the printed overview above to choose a record set and field by `@id` for further analysis.

In [ ]:
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Print DataFrame columns for the first record set (as an example)
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Fields (as columns) for RecordSet '{first_rs_id}':\n", dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

Let's select one record set and numeric field for demonstration. *Fill in the field `@id`s from the output above as appropriate!*

In [ ]:
# Example: Choose the protein quantification record set and a numeric field
record_set_id = record_set_ids[0]   # Update this with the record set @id you want to analyze

# View all columns to choose a numeric field
print("Fields in this DataFrame:", dataframes[record_set_id].columns.tolist())
# Update the following line with a valid numeric field from your chosen RecordSet
numeric_field = dataframes[record_set_id].columns[2] if len(dataframes[record_set_id].columns) > 2 else None  # example: 'cr:abundance'

threshold = 10
if numeric_field is not None and pd.api.types.is_numeric_dtype(dataframes[record_set_id][numeric_field]):
    filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (count: {len(filtered_df)}):")
    display(filtered_df.head())

    # Normalization
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Example grouping
    group_field = dataframes[record_set_id].columns[1] if len(dataframes[record_set_id].columns) > 1 else None
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field chosen. Please set 'numeric_field' to a numeric column suitable for EDA.")

## 5. Visualization
Visualize distributions or relationships using matplotlib or seaborn. For demonstration, we'll plot the chosen numeric field (if any).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id in dataframes and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[record_set_id][numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in RecordSet '{record_set_id}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
else:
    print("No suitable numeric field selected for visualization.")

## 6. Conclusion
This notebook demonstrated loading, overviewing, and basic exploration of the *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells* dataset using `mlcroissant`.

- The dataset provides protein-related mass spectrometry data via a FAIR Croissant schema.
- It is structured into multiple record sets and fields, with data accessible and analyzable in pandas DataFrames.
- Basic EDA such as filtering, normalization, grouping, and visualization can be applied readily.

For deeper analysis, refer to individual record set and field `@id`s discovered above.